# CareerLens v4.0 — Offline Semantic ATS & Resume Matcher (Fine-Tuning + INT8 ONNX Quantization)

This Colab notebook fine-tunes an ultra-compact embedding model (`TaylorAI/bge-micro-v2`, **17.4M params**, or `sentence-transformers/all-MiniLM-L6-v2`) on:
1. **Tech Stack Synonyms & Acronyms** (`K8s <-> Kubernetes`, `Next.js <-> React`, `CI/CD <-> DevOps pipeline`, `Jetpack Compose <-> Declarative Android UI`)
2. **Malaysian Job Market & Statutory Taxonomy** (`KWSP <-> EPF`, `LHDN PCB <-> MTD`, `DuitNow/FPX <-> PayNet`, `Klang Valley/Bangsar South/PJ`)
3. **Hard-Negative Pairs** (`Java != JavaScript`, `LHDN PCB payroll != Hardware PCB soldering`, `Manual QA != Automated JUnit`)

Then exports and applies **Dynamic Per-Channel INT8 Quantization** (`careerlens_ats_int8.onnx`, ~17 MB) ready for `app/src/main/assets/models/`.


In [ ]:
# Step 1: Select Base Model & Verify GPU Runtime
import os, torch
os.environ["CAREERLENS_BASE_MODEL"] = "TaylorAI/bge-micro-v2"  # Switch to "sentence-transformers/all-MiniLM-L6-v2" if desired
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!pip install -q sentence-transformers datasets onnx onnxruntime onnxscript


In [ ]:
"""
CareerLens v4.0 — Semantic ATS Embedding Fine-Tuning & INT8 ONNX Quantization Script
Runs on Google Colab GPU (T4 / L4).
Fine-tunes TaylorAI/bge-micro-v2 on:
  1. Tech stack acronym & conceptual synonym pairs (K8s <-> Kubernetes, Next.js <-> React, CI/CD <-> DevOps)
  2. Malaysian job market & statutory taxonomy (KWSP/EPF, LHDN PCB, SOCSO/EIS, DuitNow/FPX, Klang Valley/PG/JB)
  3. Full resume bullet <-> JobStreet/LinkedIn requirement sentence pairs
  4. Hard negatives (Java != JavaScript, Manual QA != Automated CI testing, Hardware PCB != LHDN PCB payroll)
Trains in two stages: CoSENTLoss (margin ranking) + CosineSimilarityLoss (percentage calibration),
exports to FP32 ONNX, applies Dynamic INT8 Quantization (QuantType.QInt8), and verifies FP32 vs INT8 fidelity.
"""

from __future__ import annotations

import inspect
import itertools
import json
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

STATUS_PATH = Path("/content/ATS_STATUS.json")
OUT_DIR = Path("/content/ats_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def write_status(state: str, step: str = "", **extra):
    payload = {"status": state, "step": step, "ts": int(time.time()), **extra}
    STATUS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"[STATUS] {state} | {step}", flush=True)


def ensure_packages():
    write_status("RUNNING", "Checking Python packages")
    need = []
    for mod, pkg in (
        ("sentence_transformers", "sentence-transformers"),
        ("datasets", "datasets"),
        ("onnx", "onnx"),
        ("onnxruntime", "onnxruntime"),
        ("onnxscript", "onnxscript"),
    ):
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print(f"Installing missing packages: {need}", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *need],
            check=True,
        )


def build_curated_datasets(seed: int = 42):
    rng = random.Random(seed)

    synonym_clusters = [
        # DevOps / Cloud / Containers
        ["K8s", "Kubernetes", "Kubernetes container orchestration", "EKS GKE cluster management", "Helm Kubernetes deployment"],
        ["CI/CD", "DevOps pipeline", "automated build and deployment pipeline", "GitHub Actions GitLab CI Jenkins", "continuous integration continuous delivery"],
        ["Docker", "containerization", "Docker containers Dockerfile", "OCI container packaging"],
        ["Terraform", "Infrastructure as Code", "IaC automated cloud provisioning", "Terraform cloud modules"],
        ["AWS", "Amazon Web Services cloud", "EC2 S3 Lambda RDS AWS architecture", "AWS cloud native services"],
        ["GCP", "Google Cloud Platform", "Cloud Run GKE BigQuery Pub/Sub GCP"],
        ["Observability", "Prometheus and Grafana metrics", "Datadog APM tracing and alerting", "application logging monitoring dashboards"],
        ["SRE", "Site Reliability Engineering", "production incident SLA reliability engineering", "on-call observability chaos resilience"],

        # Frontend / Web
        ["Next.js", "React", "React.js frontend web framework", "Next.js server-side rendering SSR", "React TypeScript SPA development"],
        ["TypeScript", "typed JavaScript", "TypeScript static typing web frontend", "TSX strongly typed JS"],
        ["Tailwind CSS", "utility-first CSS styling", "modern responsive CSS design system"],
        ["GraphQL", "Apollo GraphQL client", "typed GraphQL API schema queries"],

        # Android / Mobile / Kotlin
        ["Jetpack Compose", "declarative Android UI", "Material 3 Compose UI toolkit", "modern Android declarative UI components"],
        ["Kotlin Coroutines", "asynchronous Kotlin Flow StateFlow", "reactive concurrency multithreading in Kotlin", "structured concurrency StateFlow SharedFlow"],
        ["Room SQLite", "offline-first local mobile database", "SQLite DAO local persistence caching Android", "on-device relational database storage"],
        ["Hilt", "Dagger Hilt dependency injection", "DI inversion of control Android Koin", "modular dependency injection container"],
        ["WorkManager", "scheduled background task execution Android", "reliable deferrable offline worker jobs"],
        ["MVVM", "MVI unidirectional data flow", "Clean Architecture layered mobile architecture", "ViewModel StateFlow separation of concerns"],
        ["Google ML Kit OCR", "on-device optical character recognition", "offline text extraction computer vision mobile", "screenshot bounding-box text scanner"],
        ["ONNX Runtime Mobile", "on-device neural network inference", "quantized INT8 TFLite ONNX edge AI mobile", "offline mobile ML model deployment"],
        ["App Performance", "ANR and cold startup optimization", "Baseline Profiles Perfetto memory leak profiling", "mobile frame rate UI jank optimization"],
        ["Flutter", "Dart cross-platform mobile framework", "Flutter Bloc Riverpod multi-platform app"],
        ["React Native", "cross-platform JavaScript mobile app", "Expo React Native iOS Android"],
        ["SwiftUI", "native iOS mobile development", "Swift Combine UIKit Apple mobile engineering"],

        # Backend / APIs / Data
        ["Spring Boot", "Java Kotlin enterprise backend framework", "JVM REST microservice framework Spring"],
        ["REST API", "RESTful HTTP web services", "JSON OpenAPI Swagger endpoint design", "Retrofit OkHttp Ktor HTTP client"],
        ["gRPC", "Protocol Buffers binary RPC", "high-throughput low-latency microservice RPC"],
        ["Microservices", "distributed backend service architecture", "scalable event-driven distributed systems"],
        ["Kafka", "event streaming message broker", "RabbitMQ asynchronous pub-sub queue", "distributed event-driven architecture"],
        ["PostgreSQL", "relational SQL database", "MySQL query indexing ACID transactions", "RDBMS schema optimization"],
        ["Redis", "in-memory distributed cache", "Memcached low-latency key-value caching"],
        ["FastAPI", "Python asynchronous backend web API", "Python pydantic REST microservice"],

        # Security / QA / Process
        ["OAuth2 OIDC", "JWT token authentication authorization", "SSO identity access management security", "RBAC role-based access control"],
        ["Mobile Security", "Android Keystore hardware encryption", "AES-256 encrypted local storage biometric auth"],
        ["Automated Testing", "JUnit MockK Espresso unit and UI tests", "automated regression testing TDD", "CI automated test suite verification"],
        ["Agile Scrum", "iterative sprint delivery Kanban", "cross-functional product squad agile workflow"],

        # Malaysian Market & Statutory Taxonomy (JobStreet / LinkedIn MY / Hiredly)
        ["EPF", "KWSP", "Malaysian statutory retirement provident fund contribution", "16% employer KWSP EPF contribution benefit"],
        ["LHDN PCB", "MTD Potongan Cukai Bulanan", "Malaysian monthly income tax payroll deduction", "resident personal tax relief Zakat Malaysia"],
        ["SOCSO EIS", "PERKESO and Employment Insurance System", "Malaysia statutory employee social security"],
        ["DuitNow FPX", "PayNet Malaysian real-time payment rails", "interbank IBG RENTAS digital payment gateway Malaysia", "Malaysian e-wallet TNG GrabPay payment integration"],
        ["BNM RMiT e-KYC", "Bank Negara Malaysia digital banking compliance", "licensed Malaysia digital bank GXBank Boost Bank AEON Bank"],
        ["Klang Valley", "Kuala Lumpur KL Sentral Bangsar South", "Petaling Jaya Bandar Utama Damansara Subang", "Greater KL corporate tech hub Malaysia"],
        ["Penang tech hub", "Bayan Lepas George Town Penang", "northern Malaysia semiconductor software hub"],
        ["JB Singapore corridor", "Johor Bahru Iskandar Puteri tech hub", "cross-border SG SGD commute career track"],
        ["Notice period buyout", "annual leave offset early release", "forfeited unvested bonus sign-on buyout negotiation"],
    ]

    sentence_pairs = [
        ("Architected an offline-first Room SQLite persistence layer with Coroutines and StateFlow.",
         "Experience designing local mobile data storage, caching, and reactive async Kotlin streams.", 0.91),
        ("Built automated GitHub Actions workflows for building, linting, and releasing Android APKs.",
         "Hands-on CI/CD DevOps automation experience for mobile release engineering.", 0.91),
        ("Deployed containerized Python and Go microservices onto production Kubernetes clusters with Helm.",
         "Strong background managing K8s workloads, Docker containers, and cloud orchestration.", 0.92),
        ("Developed responsive SaaS dashboards using Next.js, TypeScript, and Tailwind CSS.",
         "Proficiency in modern React.js frontend engineering, typed JS, and responsive web UI.", 0.90),
        ("Optimized Android cold start time by 45% using Baseline Profiles and Perfetto traces.",
         "Proven track record in mobile app vitals, startup latency optimization, and profiling.", 0.90),
        ("Implemented DuitNow QR and FPX checkout flows complying with BNM RMiT guidelines.",
         "Malaysian fintech payment gateway integration and digital banking regulatory experience.", 0.91),
        ("Designed Clean Architecture MVI modules decoupled with Hilt dependency injection.",
         "Deep knowledge of scalable multi-module Android architecture and DI patterns.", 0.89),
        ("Package includes 16% employer KWSP contribution, 22 days AL, and 2-month contractual bonus.",
         "Competitive Malaysian total compensation with above-statutory EPF, generous annual leave, and bonus.", 0.90),
        ("Led cross-functional agile sprints writing MockK unit tests and Jetpack Compose UI tests.",
         "Collaborates in Scrum teams with strong automated testing discipline and declarative UI skills.", 0.89),
        ("Integrated on-device ML Kit text recognition and quantized ONNX embeddings completely offline.",
         "Mobile edge AI experience deploying compact neural models without cloud dependency.", 0.91),
        ("Configured Redis cluster caching in front of PostgreSQL to cut p99 API latency by 60%.",
         "Backend scalability expertise in relational SQL tuning and in-memory caching layers.", 0.90),
        ("Built OAuth2 and biometric hardware Keystore authentication for mobile banking.",
         "Strong application security background covering token auth and encrypted mobile storage.", 0.89),
    ]

    hard_negatives = [
        ("Java", "JavaScript", 0.08),
        ("Java Spring Boot backend", "JavaScript React frontend web", 0.14),
        ("React.js web framework", "React Native iOS Android mobile", 0.30),
        ("Manual exploratory click testing", "Automated JUnit MockK CI unit testing", 0.12),
        ("LHDN PCB monthly income tax payroll", "Printed circuit board PCB hardware soldering", 0.04),
        ("Flutter Dart mobile app", "Native Kotlin Jetpack Compose Android", 0.25),
        ("WordPress PHP brochure site", "Kubernetes microservices cloud orchestration", 0.04),
        ("Microsoft Excel spreadsheet macros", "Apache Kafka distributed streaming pipeline", 0.05),
        ("Customer support ticketing desk", "Site Reliability Engineering SRE DevOps", 0.07),
        ("Hardware PLC ladder logic", "REST API OpenAPI JSON microservice", 0.05),
        ("Photoshop graphic brochure design", "GraphQL Apollo API schema design", 0.06),
        ("Angular TypeScript web portal", "Native SwiftUI iOS mobile app", 0.14),
        ("Sales cold calling B2B outreach", "Terraform Infrastructure as Code provisioning", 0.04),
        ("Singapore CPF employer contribution", "Malaysia KWSP EPF Akaun Persaraan", 0.28),
        ("Accounting ledger audit bookkeeping", "Android Room SQLite offline mobile database", 0.07),
        ("PHP Laravel monolithic CMS", "ONNX Runtime INT8 mobile AI quantization", 0.05),
        ("Helpdesk IT printer troubleshooting", "Kubernetes container orchestration K8s", 0.05),
        ("C++ embedded microcontroller firmware", "Next.js SSR React frontend", 0.07),
    ]

    # Unseen phrasing generalization eval + CareerLens headline benchmark pairs
    unseen_eval = [
        ("K8s workloads management", "Kubernetes orchestration engine", 0.92),
        ("Next.js SSR web application", "React.js frontend UI framework", 0.90),
        ("CI/CD release pipeline", "DevOps build automation workflow", 0.91),
        ("Jetpack Compose screens", "Declarative Android UI components", 0.92),
        ("Room SQLite persistence", "Offline local mobile database cache", 0.91),
        ("KWSP employer rate", "Malaysia statutory EPF retirement contribution", 0.92),
        ("DuitNow QR checkout", "PayNet Malaysian real-time payment rail", 0.91),
        ("Bangsar South / PJ", "Klang Valley Greater KL tech hub", 0.89),
        ("Java JVM service", "JavaScript browser scripting", 0.10),
        ("PCB payroll tax deduction", "Hardware PCB circuit board soldering", 0.05),
        ("Manual UAT clicking", "Automated JUnit MockK test suite", 0.14),
    ]

    headline_probes = [
        ("K8s", "Kubernetes", 0.95),
        ("Next.js", "React", 0.92),
        ("CI/CD", "DevOps pipeline", 0.93),
        ("Jetpack Compose", "Declarative Android UI", 0.93),
        ("KWSP", "EPF", 0.95),
        ("Java", "JavaScript", 0.08),
    ]

    unseen_set = {(a.lower(), b.lower()) for a, b, _ in unseen_eval} | {(b.lower(), a.lower()) for a, b, _ in unseen_eval}

    pairs = []
    for cluster in synonym_clusters:
        for a, b in itertools.combinations(cluster, 2):
            score = 0.93 if (len(a) < 16 or len(b) < 16) else 0.90
            pairs.append((a, b, score))
            pairs.append((b, a, score))
            pairs.append((f"Skill: {a}", f"Requirement: {b}", score))

    for a, b, s in sentence_pairs:
        pairs.append((a, b, s))
        pairs.append((b, a, s))

    for _ in range(3):
        for a, b, s in hard_negatives:
            pairs.append((a, b, s))
            pairs.append((b, a, s))

    # Cross-cluster negatives
    for _ in range(360):
        c1, c2 = rng.sample(range(len(synonym_clusters)), 2)
        # Avoid pairing strongly adjacent sub-clusters (e.g. Compose vs MVVM, or DuitNow vs BNM)
        if abs(c1 - c2) <= 2:
            continue
        t1 = rng.choice(synonym_clusters[c1])
        t2 = rng.choice(synonym_clusters[c2])
        pairs.append((t1, t2, 0.08))

    train_rows = [
        {"sentence1": a, "sentence2": b, "score": float(s)}
        for a, b, s in pairs
        if (a.lower(), b.lower()) not in unseen_set
    ]
    rng.shuffle(train_rows)

    eval_rows = [
        {"sentence1": a, "sentence2": b, "score": float(s)}
        for a, b, s in (unseen_eval + headline_probes)
    ]
    return train_rows, eval_rows, len(unseen_eval)


def extract_pooling_config(st_model) -> dict:
    pooling_module = None
    for mod in st_model:
        if "pooling" in mod.__class__.__name__.lower():
            pooling_module = mod
            break
    cfg = pooling_module.get_config_dict() if pooling_module is not None else {"pooling_mode_mean_tokens": True}
    if cfg.get("pooling_mode_cls_token", False):
        mode = "cls"
    elif cfg.get("pooling_mode_max_tokens", False):
        mode = "max"
    else:
        mode = "mean"
    return {"pooling_mode": mode, "raw_pooling_config": cfg}


def np_pool_and_normalize(last_hidden_state, attention_mask, pooling_mode: str):
    import numpy as np

    if pooling_mode == "cls":
        pooled = last_hidden_state[:, 0, :]
    else:
        mask = np.expand_dims(attention_mask.astype(np.float32), axis=-1)
        summed = np.sum(last_hidden_state * mask, axis=1)
        counts = np.clip(np.sum(mask, axis=1), a_min=1e-9, a_max=None)
        pooled = summed / counts

    norms = np.linalg.norm(pooled, axis=1, keepdims=True)
    norms = np.clip(norms, a_min=1e-12, a_max=None)
    return pooled / norms


def evaluate_st_probes(st_model, eval_rows, unseen_count: int):
    import numpy as np

    s1 = [r["sentence1"] for r in eval_rows]
    s2 = [r["sentence2"] for r in eval_rows]
    e1 = st_model.encode(s1, normalize_embeddings=True)
    e2 = st_model.encode(s2, normalize_embeddings=True)
    sims = np.sum(e1 * e2, axis=1).tolist()

    unseen_rows = eval_rows[:unseen_count]
    unseen_sims = sims[:unseen_count]
    pos_sims = [sim for sim, r in zip(unseen_sims, unseen_rows) if r["score"] >= 0.7]
    neg_sims = [sim for sim, r in zip(unseen_sims, unseen_rows) if r["score"] <= 0.3]

    return {
        "unseen_positive_mean": round(float(sum(pos_sims) / max(1, len(pos_sims))), 4),
        "unseen_hard_negative_mean": round(float(sum(neg_sims) / max(1, len(neg_sims))), 4),
        "unseen_margin": round(
            float(sum(pos_sims) / max(1, len(pos_sims)) - sum(neg_sims) / max(1, len(neg_sims))),
            4,
        ),
        "pairs": [
            {
                "split": "unseen" if idx < unseen_count else "headline",
                "a": r["sentence1"],
                "b": r["sentence2"],
                "target": r["score"],
                "cosine": round(float(sim), 4),
            }
            for idx, (r, sim) in enumerate(zip(eval_rows, sims))
        ],
    }


def export_and_quantize_onnx(st_model, pooling_info: dict, eval_rows, unseen_count: int, base_model_id: str):
    import numpy as np
    import onnx
    import onnxruntime as ort
    import torch
    from onnxruntime.quantization import QuantType, quantize_dynamic

    write_status("RUNNING", "Exporting fine-tuned encoder to ONNX (FP32) + Dynamic INT8 (QInt8)")

    ft_dir = OUT_DIR / "finetuned_st"
    st_model.save_pretrained(str(ft_dir))

    tok = st_model.tokenizer
    vocab_files = tok.save_vocabulary(str(OUT_DIR))
    vocab_path = OUT_DIR / "vocab.txt"
    if not vocab_path.exists() and vocab_files:
        shutil.copy2(vocab_files[0], vocab_path)

    base_model = st_model[0].auto_model.to("cpu").eval()
    sig_params = set(inspect.signature(base_model.forward).parameters.keys())
    uses_token_type_ids = "token_type_ids" in sig_params and getattr(base_model.config, "type_vocab_size", 2) > 0

    class EncoderOnly(torch.nn.Module):
        def __init__(self, inner, with_tti: bool):
            super().__init__()
            self.inner = inner
            self.with_tti = with_tti

        def forward(self, input_ids, attention_mask, token_type_ids=None):
            if self.with_tti:
                out = self.inner(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            else:
                out = self.inner(input_ids=input_ids, attention_mask=attention_mask)
            return out.last_hidden_state

    wrapper = EncoderOnly(base_model, uses_token_type_ids).eval()
    sample_inputs = tok(
        ["Kubernetes container orchestration", "Jetpack Compose declarative UI"],
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )

    if uses_token_type_ids and "token_type_ids" in sample_inputs:
        input_tuple = (
            sample_inputs["input_ids"],
            sample_inputs["attention_mask"],
            sample_inputs["token_type_ids"],
        )
        input_names = ["input_ids", "attention_mask", "token_type_ids"]
        dynamic_axes = {
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "token_type_ids": {0: "batch_size", 1: "sequence_length"},
            "last_hidden_state": {0: "batch_size", 1: "sequence_length"},
        }
    else:
        uses_token_type_ids = False
        input_tuple = (sample_inputs["input_ids"], sample_inputs["attention_mask"])
        input_names = ["input_ids", "attention_mask"]
        dynamic_axes = {
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "last_hidden_state": {0: "batch_size", 1: "sequence_length"},
        }

    fp32_onnx = OUT_DIR / "careerlens_ats_fp32.onnx"
    export_kwargs = {
        "export_params": True,
        "opset_version": 17,
        "do_constant_folding": True,
        "input_names": input_names,
        "output_names": ["last_hidden_state"],
        "dynamic_axes": dynamic_axes,
    }
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        export_kwargs["dynamo"] = False

    with torch.no_grad():
        torch.onnx.export(wrapper, input_tuple, str(fp32_onnx), **export_kwargs)

    onnx.checker.check_model(onnx.load(str(fp32_onnx)))

    int8_onnx = OUT_DIR / "careerlens_ats_int8.onnx"
    quantize_dynamic(
        model_input=str(fp32_onnx),
        model_output=str(int8_onnx),
        per_channel=True,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gather", "Gemm"],
    )

    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = 4
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(int8_onnx), sess_opts, providers=["CPUExecutionProvider"])

    def encode_ort(texts):
        enc = tok(texts, padding=True, truncation=True, max_length=128, return_tensors="np")
        feed = {
            "input_ids": enc["input_ids"].astype(np.int64),
            "attention_mask": enc["attention_mask"].astype(np.int64),
        }
        if uses_token_type_ids:
            tti = enc.get("token_type_ids")
            if tti is None:
                tti = np.zeros_like(enc["input_ids"], dtype=np.int64)
            feed["token_type_ids"] = tti.astype(np.int64)
        out = session.run(None, feed)[0]
        return np_pool_and_normalize(out, enc["attention_mask"], pooling_info["pooling_mode"])

    s1 = [r["sentence1"] for r in eval_rows]
    s2 = [r["sentence2"] for r in eval_rows]

    pt_e1 = st_model.encode(s1, normalize_embeddings=True)
    ort_e1 = encode_ort(s1)
    ort_e2 = encode_ort(s2)

    fp32_vs_int8_cos = np.sum(pt_e1 * ort_e1, axis=1)
    int8_sims = np.sum(ort_e1 * ort_e2, axis=1).tolist()

    unseen_rows = eval_rows[:unseen_count]
    unseen_sims = int8_sims[:unseen_count]
    pos_sims = [sim for sim, r in zip(unseen_sims, unseen_rows) if r["score"] >= 0.7]
    neg_sims = [sim for sim, r in zip(unseen_sims, unseen_rows) if r["score"] <= 0.3]

    bench_enc = tok(
        ["Senior Android Engineer Kotlin Jetpack Compose Coroutines"],
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="np",
    )
    bench_feed = {
        "input_ids": bench_enc["input_ids"].astype(np.int64),
        "attention_mask": bench_enc["attention_mask"].astype(np.int64),
    }
    if uses_token_type_ids:
        bench_feed["token_type_ids"] = bench_enc["token_type_ids"].astype(np.int64)

    for _ in range(5):
        session.run(None, bench_feed)
    t_bench = time.perf_counter()
    for _ in range(50):
        session.run(None, bench_feed)
    avg_cpu_ms = ((time.perf_counter() - t_bench) / 50.0) * 1000.0

    fp32_mb = fp32_onnx.stat().st_size / (1024 * 1024)
    int8_mb = int8_onnx.stat().st_size / (1024 * 1024)

    metadata = {
        "model_id": "careerlens-bge-micro-v2-my-ats-int8",
        "base_model": base_model_id,
        "embedding_dim": int(st_model.get_sentence_embedding_dimension()),
        "max_seq_length": 128,
        "pooling_mode": pooling_info["pooling_mode"],
        "normalize_embeddings": True,
        "uses_token_type_ids": uses_token_type_ids,
        "do_lower_case": bool(getattr(tok, "do_lower_case", True)),
        "cls_token_id": int(tok.cls_token_id or 101),
        "sep_token_id": int(tok.sep_token_id or 102),
        "pad_token_id": int(tok.pad_token_id or 0),
        "unk_token_id": int(tok.unk_token_id or 100),
        "vocab_size": int(len(tok)),
        "input_names": input_names,
        "output_name": "last_hidden_state",
        "quantization": "Dynamic INT8 Per-Channel (MatMul+Gather QInt8)",
        "fp32_size_mb": round(fp32_mb, 2),
        "int8_size_mb": round(int8_mb, 2),
        "compression_ratio": round(fp32_mb / max(0.01, int8_mb), 2),
        "fp32_to_int8_mean_cosine_fidelity": round(float(np.mean(fp32_vs_int8_cos)), 5),
        "cpu_4thread_latency_64tok_ms": round(avg_cpu_ms, 2),
    }
    (OUT_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    # Remove intermediate FP32 onnx + checkpoints to keep workspace clean
    fp32_onnx.unlink(missing_ok=True)
    shutil.rmtree(ft_dir, ignore_errors=True)

    int8_eval = {
        "unseen_positive_mean": round(float(sum(pos_sims) / max(1, len(pos_sims))), 4),
        "unseen_hard_negative_mean": round(float(sum(neg_sims) / max(1, len(neg_sims))), 4),
        "unseen_margin": round(
            float(sum(pos_sims) / max(1, len(pos_sims)) - sum(neg_sims) / max(1, len(neg_sims))),
            4,
        ),
        "pairs": [
            {
                "split": "unseen" if idx < unseen_count else "headline",
                "a": r["sentence1"],
                "b": r["sentence2"],
                "target": r["score"],
                "int8_cosine": round(float(sim), 4),
            }
            for idx, (r, sim) in enumerate(zip(eval_rows, int8_sims))
        ],
    }
    return metadata, int8_eval


def make_training_args(output_dir: str, epochs: int, use_fp16: bool, lr: float):
    from sentence_transformers import SentenceTransformerTrainingArguments

    arg_sig = set(inspect.signature(SentenceTransformerTrainingArguments.__init__).parameters.keys())
    kwargs = {
        "output_dir": output_dir,
        "num_train_epochs": epochs,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "learning_rate": lr,
        "warmup_ratio": 0.1,
        "fp16": use_fp16,
        "bf16": False,
        "logging_steps": 50,
        "save_strategy": "no",
        "report_to": "none",
    }
    if "eval_strategy" in arg_sig:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in arg_sig:
        kwargs["evaluation_strategy"] = "epoch"
    return SentenceTransformerTrainingArguments(**kwargs)


def main():
    t0 = time.perf_counter()
    ensure_packages()

    import torch
    from datasets import Dataset
    from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses

    device = "cuda" if torch.cuda.is_available() else "cpu"
    gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
    print(f"Runtime device: {device} ({gpu_name})", flush=True)

    base_model_id = os.environ.get("CAREERLENS_BASE_MODEL", "TaylorAI/bge-micro-v2")
    write_status("RUNNING", f"Loading base model {base_model_id}", gpu=gpu_name)
    model = SentenceTransformer(base_model_id, device=device)
    pooling_info = extract_pooling_config(model)
    print(f"Detected pooling config: {pooling_info}", flush=True)

    train_rows, eval_rows, unseen_count = build_curated_datasets(seed=42)
    print(f"Dataset: train={len(train_rows)} pairs | unseen_eval={unseen_count} | total_eval={len(eval_rows)}", flush=True)

    pre_eval = evaluate_st_probes(model, eval_rows, unseen_count)
    print(f"Pre-Fine-Tune: pos={pre_eval['unseen_positive_mean']} neg={pre_eval['unseen_hard_negative_mean']} margin={pre_eval['unseen_margin']}", flush=True)

    train_dataset = Dataset.from_list(train_rows)
    eval_dataset = Dataset.from_list(eval_rows)

    # Stage 1: CoSENTLoss (sharp ranking separation)
    write_status("RUNNING", "Stage 1: CoSENTLoss ranking fine-tuning (6 epochs)")
    trainer_s1 = SentenceTransformerTrainer(
        model=model,
        args=make_training_args("/content/ats_ckpt_s1", epochs=6, use_fp16=(device == "cuda"), lr=4e-5),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=losses.CoSENTLoss(model=model),
    )
    trainer_s1.train()

    # Stage 2: CosineSimilarityLoss (calibrate absolute similarity percentages to [0.05..0.95])
    write_status("RUNNING", "Stage 2: CosineSimilarityLoss score calibration (10 epochs)")
    trainer_s2 = SentenceTransformerTrainer(
        model=model,
        args=make_training_args("/content/ats_ckpt_s2", epochs=10, use_fp16=(device == "cuda"), lr=3e-5),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=losses.CosineSimilarityLoss(model=model),
    )
    trainer_s2.train()

    post_eval = evaluate_st_probes(model, eval_rows, unseen_count)
    print(f"Post-Fine-Tune FP32: pos={post_eval['unseen_positive_mean']} neg={post_eval['unseen_hard_negative_mean']} margin={post_eval['unseen_margin']}", flush=True)

    metadata, int8_eval = export_and_quantize_onnx(model, pooling_info, eval_rows, unseen_count, base_model_id)
    print(f"Post-Quant INT8 ONNX: pos={int8_eval['unseen_positive_mean']} neg={int8_eval['unseen_hard_negative_mean']} margin={int8_eval['unseen_margin']}", flush=True)

    elapsed = round(time.perf_counter() - t0, 2)
    report = {
        "gpu": gpu_name,
        "elapsed_seconds": elapsed,
        "train_pairs": len(train_rows),
        "unseen_eval_pairs": unseen_count,
        "model_metadata": metadata,
        "pre_finetune_eval": pre_eval,
        "post_finetune_fp32_eval": post_eval,
        "post_quantization_int8_eval": int8_eval,
    }
    (OUT_DIR / "careerlens_eval_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

    artifacts = [p.name for p in sorted(OUT_DIR.iterdir()) if p.is_file()]
    write_status("DONE", "Complete", elapsed_seconds=elapsed, artifacts=artifacts)
    print(f"=== FINISHED IN {elapsed}s | Artifacts: {artifacts} ===", flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        import traceback
        tb = traceback.format_exc()
        print(tb, flush=True)
        write_status("ERROR", "Failed", error=str(exc), tb=tb[-2500:])
        raise


In [ ]:
# Step 3: Print Evaluation Summary (Pre-Fine-Tune vs Fine-Tuned FP32 vs Quantized INT8 ONNX)
import json
from pathlib import Path

report = json.loads(Path("/content/ats_outputs/careerlens_eval_report.json").read_text())
print("Metadata:", json.dumps(report["model_metadata"], indent=2))
print("\nPre-Fine-Tune Margin :", report["pre_finetune_eval"]["unseen_margin"])
print("Post-Fine-Tune Margin:", report["post_finetune_fp32_eval"]["unseen_margin"])
print("Quantized INT8 Margin:", report["post_quantization_int8_eval"]["unseen_margin"])


In [ ]:
# Step 4 (Optional): Export .tflite via Google ai-edge-torch OR package assets ZIP for CareerLens
EXPORT_TFLITE = False
if EXPORT_TFLITE:
    !pip install -q ai-edge-torch
    import ai_edge_torch, torch
    from transformers import AutoModel
    # Export fine-tuned checkpoint if converted from sentence_transformers
    base_m = AutoModel.from_pretrained(os.environ["CAREERLENS_BASE_MODEL"]).eval()
    sample_ids = torch.ones((1, 128), dtype=torch.long)
    sample_mask = torch.ones((1, 128), dtype=torch.long)
    edge_model = ai_edge_torch.convert(base_m, (sample_ids, sample_mask))
    edge_model.export("/content/ats_outputs/careerlens_ats.tflite")
    print("Exported /content/ats_outputs/careerlens_ats.tflite")

!zip -j /content/careerlens_models_pack.zip /content/ats_outputs/careerlens_ats_int8.onnx /content/ats_outputs/vocab.txt /content/ats_outputs/model_metadata.json
!ls -lh /content/ats_outputs /content/careerlens_models_pack.zip
